In [3]:
import random


class DealerManager:
    """
    Handles dealer selection.
    - First dealer is chosen randomly
    - Dealer moves left each new round
    """

    def __init__(self, players):
        self.players = players
        self.dealer_index = None

    def new_dealer(self):
        """Choose the first dealer randomly."""
        self.dealer_index = random.randint(0, len(self.players) - 1)
        return self.dealer_index

    def left_dealer(self):
        """Move dealer to the player on the left."""
        if self.dealer_index is None:
            raise ValueError("Dealer has not been chosen yet.")
        self.dealer_index = (self.dealer_index + 1) % len(self.players)
        return self.dealer_index

    def get_dealer_index(self):
        return self.dealer_index


class BidManager:
    """
    Handles bidding and trump suit selection.
    Rule used here:
    every player must either bid higher than the current high bid or pass with 0.
    """

    VALID_SUITS = ["clubs", "diamonds", "hearts", "spades"]

    def __init__(self, players):
        self.players = players
        self.reset_round_bidding()

    def reset_round_bidding(self):
        """Reset all bidding data for a new round."""
        self.bids = {player: 0 for player in self.players}
        self.high_bid = 0
        self.high_bidder_index = None
        self.high_bidder_name = None
        self.trump_suit = None

    def validate_bid(self, bid, current_high_bid):
        """
        Validate any bid.
        Rules:
        - bid must be an integer
        - bid must be between 0 and 18
        - bid must either be 0 (pass) or greater than current_high_bid
        """
        if not isinstance(bid, int):
            raise ValueError("Bid must be an integer.")

        if bid < 0 or bid > 18:
            raise ValueError("Bid must be between 0 and 18.")

        if bid != 0 and bid <= current_high_bid:
            raise ValueError(
                f"Bid must be higher than the current high bid ({current_high_bid}) or 0 to pass."
            )

        return bid

    def validate_player_bid(self, bid, current_high_bid):
        """
        Validate the human player's bid.
        """
        return self.validate_bid(bid, current_high_bid)

    def get_bot_bid(self, current_high_bid):
        """
        Give the bot a random valid bid.

        The bot can only:
        - pass with 0
        - or bid higher than the current high bid
        """
        valid_choices = [0] + list(range(current_high_bid + 1, 19))
        return random.choice(valid_choices)

    def collect_bids(self, dealer_index, player_bid):
        """
        Run the full bidding phase.
        Turn order starts with the player to the left of the dealer.

        Each player gets one turn.
        On that turn they must either:
        - bid higher than the current high bid
        - or pass with 0
        """
        self.reset_round_bidding()

        start_index = (dealer_index + 1) % len(self.players)

        print("\n--- Bidding Phase ---")
        print(f"Dealer: {self.players[dealer_index]}")

        for turn in range(len(self.players)):
            player_index = (start_index + turn) % len(self.players)
            player_name = self.players[player_index]

            if player_index == 0:
                # Human player's turn:
                # they must beat the current high bid or pass with 0
                bid = self.validate_player_bid(player_bid, self.high_bid)
                print(f"{player_name} bids: {bid}")
            else:
                # Bot must also beat the current high bid or pass with 0
                bid = self.get_bot_bid(self.high_bid)
                bid = self.validate_bid(bid, self.high_bid)
                print(f"{player_name} bids: {bid}")

            self.bids[player_name] = bid

            if bid > self.high_bid:
                self.high_bid = bid
                self.high_bidder_index = player_index
                self.high_bidder_name = player_name

        # If everyone passes with 0, dealer is forced to 2
        if all(bid == 0 for bid in self.bids.values()):
            dealer_name = self.players[dealer_index]
            self.bids[dealer_name] = 2
            self.high_bid = 2
            self.high_bidder_index = dealer_index
            self.high_bidder_name = dealer_name
            print(f"All players passed. Dealer rule applied: {dealer_name}'s bid becomes 2.")

        return self.bids, self.high_bidder_name, self.high_bid

    def validate_trump_suit(self, suit):
        """Validate trump suit choice."""
        if not isinstance(suit, str):
            raise ValueError("Trump suit must be a string.")

        suit = suit.strip().lower()

        if suit not in self.VALID_SUITS:
            raise ValueError("Trump suit must be clubs, diamonds, hearts, or spades.")

        return suit

    def choose_trump_suit(self, player_trump_suit=None):
        """
        If the player wins, use the passed-in trump suit.
        If a bot wins, choose a random trump suit.
        """
        if self.high_bidder_index is None:
            raise ValueError("No winning bidder found. Run collect_bids() first.")

        if self.high_bidder_index == 0:
            if player_trump_suit is None:
                raise ValueError("Player won the bid, so a trump suit must be provided.")
            self.trump_suit = self.validate_trump_suit(player_trump_suit)
        else:
            self.trump_suit = random.choice(self.VALID_SUITS)
            print(f"{self.high_bidder_name} chooses trump: {self.trump_suit}")

        return self.trump_suit


class GameController:
    """
    Controls round flow by coordinating DealerManager and BidManager.
    """

    def __init__(self):
        self.players = ["Player", "Bot 1", "Bot 2", "Bot 3"]
        self.dealer_manager = DealerManager(self.players)
        self.bid_manager = BidManager(self.players)
        self.round_number = 0

    def start_game(self):
        """Start the game and choose the first dealer."""
        self.round_number = 1
        first_dealer_index = self.dealer_manager.new_dealer()
        print("--- Game Started ---")
        print(f"First dealer: {self.players[first_dealer_index]}")

    def start_round(self, player_bid, player_trump_suit=None):
        """
        Start one round of bidding and trump selection.

        player_bid:
            the player's bid for their turn
            must be higher than the current high bid when their turn arrives,
            or 0 to pass
        """
        dealer_index = self.dealer_manager.get_dealer_index()

        if dealer_index is None:
            raise ValueError("Dealer has not been selected. Call start_game() first.")

        print(f"\n=== Round {self.round_number} ===")
        print(f"Dealer: {self.players[dealer_index]}")

        bids, winning_player, winning_bid = self.bid_manager.collect_bids(dealer_index, player_bid)

        print("\nFinal Bids:")
        for player, bid in bids.items():
            print(f"{player}: {bid}")

        print(f"\nWinning bidder: {winning_player} with {winning_bid}")

        trump_suit = self.bid_manager.choose_trump_suit(player_trump_suit)
        print(f"Trump suit for this round: {trump_suit}")

        return {
            "round_number": self.round_number,
            "dealer": self.players[dealer_index],
            "bids": bids,
            "winning_bidder": winning_player,
            "winning_bid": winning_bid,
            "trump_suit": trump_suit
        }

    def end_round(self):
        """End the current round and rotate dealer left."""
        next_dealer_index = self.dealer_manager.left_dealer()
        self.round_number += 1
        print(f"\nNext dealer will be: {self.players[next_dealer_index]}")


# -------------------------
# Main program
# -------------------------
if __name__ == "__main__":
    game = GameController()
    game.start_game()

    # Example:
    # The player must either bid higher than the current high bid
    # when their turn happens, or pass with 0.
    player_bid = int(input("Enter your bid: "))
    player_trump_suit = input("Enter trump suit: ")
    round_data = game.start_round(player_bid=player_bid, player_trump_suit=player_trump_suit)

    print("\n--- Round Summary ---")
    print(round_data)

    game.end_round()

--- Game Started ---
First dealer: Bot 3


Enter your bid:  0
Enter trump suit:  diamonds



=== Round 1 ===
Dealer: Bot 3

--- Bidding Phase ---
Dealer: Bot 3
Player bids: 0
Bot 1 bids: 3
Bot 2 bids: 13
Bot 3 bids: 16

Final Bids:
Player: 0
Bot 1: 3
Bot 2: 13
Bot 3: 16

Winning bidder: Bot 3 with 16
Bot 3 chooses trump: hearts
Trump suit for this round: hearts

--- Round Summary ---
{'round_number': 1, 'dealer': 'Bot 3', 'bids': {'Player': 0, 'Bot 1': 3, 'Bot 2': 13, 'Bot 3': 16}, 'winning_bidder': 'Bot 3', 'winning_bid': 16, 'trump_suit': 'hearts'}

Next dealer will be: Player
